In [ ]:
# llm_exercise 프로젝트에서 진행

# 1. Qwen2.5-1.5B-Instruct

In [ ]:
# https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct
# 라이브러리 설치 transformers accelerate

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
###############################################################################################################
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
###############################################################################################################
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [9]:
prompt = "hi"
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt},
]
print(messages)
print("="*100)
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
print(text)
print("="*100)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
print(model_inputs)

[{'role': 'system', 'content': 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.'}, {'role': 'user', 'content': 'hi'}]
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
hi<|im_end|>
<|im_start|>assistant

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198,   6023, 151645,    198,
         151644,  77091,    198]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1]], device='cuda:0')}


In [10]:
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
print(generated_ids)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]
print(generated_ids)
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198,   6023, 151645,    198,
         151644,  77091,    198,   9707,      0,   2585,    646,    358,   1492,
            498,   3351,     30,   2160,   1052,   2494,   3151,    498,   1035,
           1075,    311,   1414,    476,   4263,     30, 151645]],
       device='cuda:0')
[tensor([  9707,      0,   2585,    646,    358,   1492,    498,   3351,     30,
          2160,   1052,   2494,   3151,    498,   1035,   1075,    311,   1414,
           476,   4263,     30, 151645], device='cuda:0')]
Hello! How can I help you today? Is there something specific you would like to know or discuss?


In [ ]:
# messages -> response 까지 나오는 과정 파악하기 
# 1_0_huggingface.ipynb에서 다룬 kanana로 다시 확인해보기


# 2. Kanana-nano-2.1b-instruct

In [ ]:
# https://huggingface.co/kakaocorp/kanana-nano-2.1b-instruct
# 모델 에러 해결 - 주로 trasnformers의 버전 문제일 가능성이 높다.
# uv remove transformers
# uv add transformers==4.45.0

In [6]:
import transformers
print(transformers.__version__)

4.50.0


In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "kakaocorp/kanana-nano-2.1b-instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
).to("cuda")
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [8]:
# 질문 바로 입력 → 답변 생성
messages = [
    {"role": "system", "content": "You are a helpful AI assistant developed by Kakao."},
    {"role": "user", "content": "한국의 수도는 어디야? 간단히 알려줘."}
]

model_inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

In [9]:
with torch.no_grad():
    output = model.generate(
        **model_inputs,
        max_new_tokens=100,
        do_sample=False,
    )

# 입력 부분 제외하고 생성된 답변만 출력
response = tokenizer.decode(output[0][model_inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print("[질문] 한국의 수도는 어디야? 간단히 알려줘.")
print(f"[답변] {response}")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[질문] 한국의 수도는 어디야? 간단히 알려줘.
[답변] 한국의 수도는 서울입니다.


# Gemma-3-1b-it

In [ ]:
# https://huggingface.co/google/gemma-3-1b-it
# 모델 불러오기 에러 해결 과정
# uv remove transformers
# uv add transformers==4.50.0
# uv add bitsandbytes

In [1]:
from huggingface_hub import login 
import os 

login(token=os.getenv("HUGGINGFACE_TOKEN"))

In [2]:
from transformers import pipeline
import torch

pipe = pipeline("text-generation", model="google/gemma-3-1b-it", device="cuda", torch_dtype=torch.bfloat16)

messages = [
    [
        {
            "role": "system",
            "content": [{"type": "text", "text": "You are a helpful assistant."},]
        },
        {
            "role": "user",
            "content": [{"type": "text", "text": "Write a poem on Hugging Face, the company"},]
        },
    ],
]

output = pipe(messages, max_new_tokens=50)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

c:\potenup3\local_llm_exercise\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--google--gemma-3-1b-it. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Device set to use cuda


In [3]:
print(output)

[[{'generated_text': [{'role': 'system', 'content': [{'type': 'text', 'text': 'You are a helpful assistant.'}]}, {'role': 'user', 'content': [{'type': 'text', 'text': 'Write a poem on Hugging Face, the company'}]}, {'role': 'assistant', 'content': 'Okay, here’s a poem about Hugging Face, aiming to capture its essence and impact:\n\n**The Neural Bloom**\n\nIn shadows deep, where models reside,\nA network stirs, with purpose tied.\nHugging Face, a'}]}]]


In [4]:
for out in output:
    print(out)

<bos><start_of_turn>user
You are a helpful assistant.

Write a poem on Hugging Face, the company<end_of_turn>
<start_of_turn>model
Okay, here's a poem about Hugging Face, aiming to capture its essence and the feeling it evokes:

**The Neural Web**

A cloud of models, vast and bright,
Hugging Face, a shining light.
A platform built for minds so keen,
To learn and build, a


In [1]:
from transformers import AutoTokenizer, BitsAndBytesConfig, Gemma3ForCausalLM
import torch

model_id = "google/gemma-3-1b-it"

quantization_config = BitsAndBytesConfig(load_in_8bit=True)

model = Gemma3ForCausalLM.from_pretrained(
    model_id, quantization_config=quantization_config
).eval()

tokenizer = AutoTokenizer.from_pretrained(model_id)


W0327 10:57:42.365000 14396 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
`low_cpu_mem_usage` was None, now default to True since model is quantized.
Attempting to cast a BatchEncoding to type torch.bfloat16. This is not supported.


In [ ]:

messages = [
    [
        {
            "role": "system",
            "content": [{"type": "text", "text": "You are a helpful assistant."},]
        },
        {
            "role": "user",
            "content": [{"type": "text", "text": "Write a poem on Hugging Face, the company"},]
        },
    ],
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device).to(torch.bfloat16)


In [ ]:


with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=64)

outputs = tokenizer.batch_decode(outputs)


In [2]:
print(outputs)

["<bos><start_of_turn>user\nYou are a helpful assistant.\n\nWrite a poem on Hugging Face, the company<end_of_turn>\n<start_of_turn>model\nOkay, here's a poem about Hugging Face, aiming to capture its essence and the feeling it evokes:\n\n**The Neural Web**\n\nA cloud of models, vast and bright,\nHugging Face, a shining light.\nA platform built for minds so keen,\nTo learn and build, a"]


In [3]:
for out in outputs:
    print(out)

<bos><start_of_turn>user
You are a helpful assistant.

Write a poem on Hugging Face, the company<end_of_turn>
<start_of_turn>model
Okay, here's a poem about Hugging Face, aiming to capture its essence and the feeling it evokes:

**The Neural Web**

A cloud of models, vast and bright,
Hugging Face, a shining light.
A platform built for minds so keen,
To learn and build, a


# 과제

In [ ]:
# 1. LLM Instruct의 모델을 Input에서 model을 거쳐 Output까지 나오는 과정을 설명해주세요
# 2. 각각의 모델마다 오류를 겪었습니다. 미리 각 모델마다 주의해야 하는 것. 추가적으로 설치해야 하는 라이브러리를 모델카드로 만들어주세요.

# 파인튜닝 데이터셋은 어떻게 생겼을까?

In [ ]:
# messages = [
#     [
#         {
#             "role": "system",
#             "content": [{"type": "text", "text": "You are a helpful assistant."},]
#         },
#         {
#             "role": "user",
#             "content": [{"type": "text", "text": "Write a poem on Hugging Face, the company"},]
#         },
#     ],
# ]

## 1) Dataset 이해하기

In [11]:
# 라이브러리 설치 uv add datasets
from datasets import load_dataset

dataset = load_dataset("beomi/KoAlpaca-v1.1a")
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'output', 'url'],
        num_rows: 21155
    })
})

In [12]:
dataset["train"]

Dataset({
    features: ['instruction', 'output', 'url'],
    num_rows: 21155
})

In [17]:
dataset["train"][0]

{'instruction': '양파는 어떤 식물 부위인가요? 그리고 고구마는 뿌리인가요?',
 'output': '양파는 잎이 아닌 식물의 줄기 부분입니다. 고구마는 식물의 뿌리 부분입니다. \n\n식물의 부위의 구분에 대해 궁금해하는 분이라면 분명 이 질문에 대한 답을 찾고 있을 것입니다. 양파는 잎이 아닌 줄기 부분입니다. 고구마는 다른 질문과 답변에서 언급된 것과 같이 뿌리 부분입니다. 따라서, 양파는 식물의 줄기 부분이 되고, 고구마는 식물의 뿌리 부분입니다.\n\n 덧붙이는 답변: 고구마 줄기도 볶아먹을 수 있나요? \n\n고구마 줄기도 식용으로 볶아먹을 수 있습니다. 하지만 줄기 뿐만 아니라, 잎, 씨, 뿌리까지 모든 부위가 식용으로 활용되기도 합니다. 다만, 한국에서는 일반적으로 뿌리 부분인 고구마를 주로 먹습니다.',
 'url': 'https://kin.naver.com/qna/detail.naver?d1id=11&dirId=1116&docId=55320268'}

## 2) datasets 기능으로 데이터셋 만들기

In [19]:
sampledata = dataset["train"].select(range(10))
sampledata

Dataset({
    features: ['instruction', 'output', 'url'],
    num_rows: 10
})

In [20]:
sampledata[0]

{'instruction': '양파는 어떤 식물 부위인가요? 그리고 고구마는 뿌리인가요?',
 'output': '양파는 잎이 아닌 식물의 줄기 부분입니다. 고구마는 식물의 뿌리 부분입니다. \n\n식물의 부위의 구분에 대해 궁금해하는 분이라면 분명 이 질문에 대한 답을 찾고 있을 것입니다. 양파는 잎이 아닌 줄기 부분입니다. 고구마는 다른 질문과 답변에서 언급된 것과 같이 뿌리 부분입니다. 따라서, 양파는 식물의 줄기 부분이 되고, 고구마는 식물의 뿌리 부분입니다.\n\n 덧붙이는 답변: 고구마 줄기도 볶아먹을 수 있나요? \n\n고구마 줄기도 식용으로 볶아먹을 수 있습니다. 하지만 줄기 뿐만 아니라, 잎, 씨, 뿌리까지 모든 부위가 식용으로 활용되기도 합니다. 다만, 한국에서는 일반적으로 뿌리 부분인 고구마를 주로 먹습니다.',
 'url': 'https://kin.naver.com/qna/detail.naver?d1id=11&dirId=1116&docId=55320268'}

In [23]:
# data가 들어왔을 때 messages = [{"role":, "content":},{"role":, "content":}] 형태로 만드는 함수 만들기
def make_prompt(data):
    return {
        "result": [
            {"role": "user", "content": data["instruction"]},
            {"role": "user", "content": data["output"]}
        ]   
    }

In [24]:
make_prompt(sampledata[0])

{'result': [{'role': 'user', 'content': '양파는 어떤 식물 부위인가요? 그리고 고구마는 뿌리인가요?'},
  {'role': 'user',
   'content': '양파는 잎이 아닌 식물의 줄기 부분입니다. 고구마는 식물의 뿌리 부분입니다. \n\n식물의 부위의 구분에 대해 궁금해하는 분이라면 분명 이 질문에 대한 답을 찾고 있을 것입니다. 양파는 잎이 아닌 줄기 부분입니다. 고구마는 다른 질문과 답변에서 언급된 것과 같이 뿌리 부분입니다. 따라서, 양파는 식물의 줄기 부분이 되고, 고구마는 식물의 뿌리 부분입니다.\n\n 덧붙이는 답변: 고구마 줄기도 볶아먹을 수 있나요? \n\n고구마 줄기도 식용으로 볶아먹을 수 있습니다. 하지만 줄기 뿐만 아니라, 잎, 씨, 뿌리까지 모든 부위가 식용으로 활용되기도 합니다. 다만, 한국에서는 일반적으로 뿌리 부분인 고구마를 주로 먹습니다.'}]}

In [25]:
test = sampledata.map(make_prompt)
test

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'output', 'url', 'result'],
    num_rows: 10
})

In [26]:
test[0]

{'instruction': '양파는 어떤 식물 부위인가요? 그리고 고구마는 뿌리인가요?',
 'output': '양파는 잎이 아닌 식물의 줄기 부분입니다. 고구마는 식물의 뿌리 부분입니다. \n\n식물의 부위의 구분에 대해 궁금해하는 분이라면 분명 이 질문에 대한 답을 찾고 있을 것입니다. 양파는 잎이 아닌 줄기 부분입니다. 고구마는 다른 질문과 답변에서 언급된 것과 같이 뿌리 부분입니다. 따라서, 양파는 식물의 줄기 부분이 되고, 고구마는 식물의 뿌리 부분입니다.\n\n 덧붙이는 답변: 고구마 줄기도 볶아먹을 수 있나요? \n\n고구마 줄기도 식용으로 볶아먹을 수 있습니다. 하지만 줄기 뿐만 아니라, 잎, 씨, 뿌리까지 모든 부위가 식용으로 활용되기도 합니다. 다만, 한국에서는 일반적으로 뿌리 부분인 고구마를 주로 먹습니다.',
 'url': 'https://kin.naver.com/qna/detail.naver?d1id=11&dirId=1116&docId=55320268',
 'result': [{'content': '양파는 어떤 식물 부위인가요? 그리고 고구마는 뿌리인가요?', 'role': 'user'},
  {'content': '양파는 잎이 아닌 식물의 줄기 부분입니다. 고구마는 식물의 뿌리 부분입니다. \n\n식물의 부위의 구분에 대해 궁금해하는 분이라면 분명 이 질문에 대한 답을 찾고 있을 것입니다. 양파는 잎이 아닌 줄기 부분입니다. 고구마는 다른 질문과 답변에서 언급된 것과 같이 뿌리 부분입니다. 따라서, 양파는 식물의 줄기 부분이 되고, 고구마는 식물의 뿌리 부분입니다.\n\n 덧붙이는 답변: 고구마 줄기도 볶아먹을 수 있나요? \n\n고구마 줄기도 식용으로 볶아먹을 수 있습니다. 하지만 줄기 뿐만 아니라, 잎, 씨, 뿌리까지 모든 부위가 식용으로 활용되기도 합니다. 다만, 한국에서는 일반적으로 뿌리 부분인 고구마를 주로 먹습니다.',
   'role': 'user'}]}